# 3D Spatial Reasoning using multiple choice

Task: Given two orthographic views (front and top) of a 3D voxel scene,
choose which of 4 options correctly shows the scene from the right side.

In [ ]:
import kaggle_benchmarks as kbench
import json
from typing import Dict
from IPython.display import display, Markdown
import pandas as pd
import re
import time

## Input constants

Input dataset has 200 tasks maximum. TASKS_SIZE selects range of tasks. Use 3 to 5 tasks for debugging, 50 or 100 tasks for benchmark evaluation.

In [ ]:
TASKS_SIZE = 50
TASK_NAME = f"multiple choice, fixed size 3" 
DATASET_VARIANT = "mc_fixed_3"

In [ ]:
# See https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# to select a different model to test
# Use `kbench.llms` to list all available models
# `kbench.llm` is the default Kaggle defined model

USE_MODEL = kbench.llm #kbench.llms["anthropic/claude-opus-4-6@default"]

## Supporting functions

In [ ]:
def format_input_views(front_view: list[list[int]], 
                       top_view: list[list[int]]) -> str:
    """
    Format both input views as text.
    
    Args:
        front_view: 2D grid for front view
        top_view: 2D grid for top view
        
    Returns:
        Formatted text showing both views
    """
    lines = []
    
    # Format front view as JSON-style array
    lines.append("FRONT VIEW (looking along +Y axis: X horizontal, Z vertical):")
    lines.append("[")
    for i, row in enumerate(front_view):
        row_str = " [" + ", ".join(str(cell) for cell in row) + "]"
        if i < len(front_view) - 1:
            row_str += ","
        lines.append(row_str)
    lines.append("]")
    
    lines.append("")
    
    # Format top view as JSON-style array
    lines.append("TOP VIEW (looking down -Z axis: X horizontal, Y vertical):")
    lines.append("[")
    for i, row in enumerate(top_view):
        row_str = " [" + ", ".join(str(cell) for cell in row) + "]"
        if i < len(top_view) - 1:
            row_str += ","
        lines.append(row_str)
    lines.append("]")
    
    return "\n".join(lines)


def format_multiple_choices(choices: Dict[str, list[list[int]]]) -> str:
    """
    Format the multiple choice options as text.
    
    Args:
        choices: Dictionary mapping choice numbers (as strings) to grids
        
    Returns:
        Formatted text showing all choices
    """
    lines = []
    lines.append("ANSWER CHOICES for RIGHT VIEW:")
    lines.append("")
    
    for choice_num in sorted(choices.keys()):
        grid = choices[choice_num]
        lines.append(f"Option {choice_num}:")
        lines.append("[")
        for i, row in enumerate(grid):
            row_str = " [" + ", ".join(str(cell) for cell in row) + "]"
            if i < len(grid) - 1:
                row_str += ","
            lines.append(row_str)
        lines.append("]")
        lines.append("")
    
    return "\n".join(lines)

In [ ]:
def build_prompt(
    front_view: list[list[int]],
    top_view: list[list[int]],
    multiple_choices: Dict[str, list[list[int]]],
) -> str:
    """
    Build the complete prompt text from task data.
    
    This helper function can be used to preview/export prompts without running the task.
    
    Args:
        front_view: 2D grid for front view
        top_view: 2D grid for top view
        multiple_choices: Dictionary mapping choice numbers to grids
        
    Returns:
        Complete formatted prompt string
    """
    # Format the input views as text
    input_text = format_input_views(front_view, top_view)
    
    # Format the multiple choices
    choices_text = format_multiple_choices(multiple_choices)
    
    # Build and return the full prompt
    return f"""
You are given the FRONT and TOP orthographic views of a 3D voxel scene. 
Each view shows the FIRST non-zero voxel encountered along the viewing direction.
Each cell contains a number (0-9). Number 0 represents an empty or transparent voxel.

Based on these FRONT VIEW and TOP VIEW, determine which option correctly shows what the scene looks like from the RIGHT VIEW.

CRITICAL OUTPUT REQUIREMENTS:
- Output ONLY a single digit: 1, 2, 3, or 4
- Do NOT include the word "Option"
- Do NOT include any explanation, reasoning, or commentary
- Do NOT use LaTeX, markdown, or formatted text
- Do NOT use phrases like "The final answer is" or similar
- Your ENTIRE response must be ONLY the digit
- If you include ANY text besides the digit, your answer will be marked INCORRECT

## Camera positions

- FRONT VIEW
    - Camera position: Y = -infinity
    - Looking direction: toward +Y
    - Projection rule: first non-zero voxel along +Y
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = lowest X (left side of the scene)

- TOP VIEW
    - Camera position: Z = +infinity
    - Looking direction: toward -Z
    - Projection rule: first non-zero voxel along -Z
    - View orientation:
        - Row 0 = highest Y (closest to back of the scene)
        - Column 0 = lowest X (left side of the scene)
        
- RIGHT VIEW
    - Camera position: X = +infinity
    - Looking direction: toward -X
    - Projection rule: first non-zero voxel along -X
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = lowest Y (left side of the scene)

{input_text}

{choices_text}

---

# EXAMPLE

Given:
FRONT VIEW:
[
  [5 0 6],
  [0 0 0],
  [1 4 2]
]
TOP VIEW:
[ 
  [7 4 8],
  [0 0 0],
  [5 0 6]
]

ANSWER CHOICES for RIGHT VIEW:
Option 1:
  [
    [8 0 6], 
    [0 0 0], 
    [4 0 2]
  ]
  
Option 2:
  [
    [6 0 8], 
    [0 0 0], 
    [4 0 2]
  ]
  
Option 3:
  [
    [6 0 8], 
    [0 0 0], 
    [2 0 5]
  ]

Option 4:
  [
    [6 0 8], 
    [0 0 0], 
    [2 0 4]
  ]

Correct answer: Option 4
Output: 4

--- 

INSTRUCTIONS:
1. Analyze the FRONT view to understand which voxels exist at each (X, Z) position
2. Analyze the TOP view to understand which voxels exist at each (X, Y) position
3. Mentally reconstruct the full 3D scene by combining both views
4. Project the 3D scene onto the Y-Z plane (the right-side view)
5. Compare your mental image with each of the 4 options
6. Select the option number that matches the correct right-side view
RESPOND WITH ONLY THE OPTION NUMBER: 1, 2, 3, or 4

DO NOT INCLUDE:
- Explanations or reasoning
- LaTeX or mathematical notation  
- Natural language text
- Phrases like "The final answer is" or "Option"
- Code blocks or markdown formatting

VALID RESPONSE EXAMPLES:
1
2
3
4

Now output your answer:
"""

In [ ]:
def strip_response(response: str) -> str:
    # Strip reasoning/thinking tags from various LLM makers
    # DeepSeek R1: <think>...</think>
    # Claude/others: <thinking>...</thinking>
    # Generic: <reasoning>...</reasoning>
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL | re.IGNORECASE)
    response = re.sub(r'<thinking>.*?</thinking>', '', response, flags=re.DOTALL | re.IGNORECASE)
    response = re.sub(r'<reasoning>.*?</reasoning>', '', response, flags=re.DOTALL | re.IGNORECASE)
    return response

In [ ]:
def extract_answer_from_response(response: str) -> int:
    """Extract the answer number (1-4) from an LLM response."""
    trimmed_response = response.strip()
    
    # If response is a single digit, check if it's valid
    if len(trimmed_response) == 1 and trimmed_response in '01234':
        return int(trimmed_response)
    
    # Find LAST occurrence of 1, 2, 3, or 4 as a standalone digit
    matches = re.findall(r'\b([1-4])\b', response)
    if matches:
        return int(matches[-1])  # Return the last match
    
    # Fallback: use judge LLM 
    try:
        extract = kbench.judge_llm.prompt(
            f"""Extract the answer number (1, 2, 3, or 4) from this response. Return ONLY the digit:

{response}"""
        )
        matches = re.findall(r'\b([1-4])\b', extract)
        if matches:
            return int(matches[-1])  # Return the last match
    except Exception as e:

        print(f"Judge LLM extraction failed: {e}")

        return 0


In [ ]:
@kbench.task(name="single_task", store_task=False)
def single_task(llm, task) -> dict:
    # Initialize base result with common fields
    result = {
        "id": task['id'],
        "model": llm.name
    }
    
    try:
        print(f"##### Evaluating task id {task['id']} with {llm.name}")
        
        # Default LLM times out causing very long wait time after update
        if llm.name == "google/gemini-2.5-flash":
            result.update({
                "predicted_answer": "SKIPPED DEFAULT LLM",
                "time_elapsed": 0.0,
                "is_correct": False
            })
            return result

            
        prompt = build_prompt(
             front_view=task['input']['front'],
             top_view=task['input']['top'],
             multiple_choices=task['multiple_choices']
         )

        # Measure LLM response time
        llm_start = time.time()
        response = llm.prompt(prompt)
        llm_elapsed = time.time() - llm_start
        print(f"LLM response time: {llm_elapsed:.1f}s")
        response = strip_response(response)
        #print(f"LLM response: {response}")  # Uncomment only during debugging
        
        # Measure answer extraction time
        extraction_start = time.time()
        predicted_answer = extract_answer_from_response(response)
        extraction_elapsed = time.time() - extraction_start
        print(f"Answer extraction time: {extraction_elapsed:.1f}s")
        # print(f"Predicted answer: {predicted_answer}")  # Uncomment only during debugging
        response = llm.prompt(prompt)
        
        is_correct = (predicted_answer == int(task['answer']))
        print(f"Is correct: {is_correct}")
        
        # Update result with computed fields
        result.update({
            "predicted_answer": predicted_answer,
            "time_elapsed": round(llm_elapsed, 1),
            "is_correct": is_correct
        })
        
    except Exception as e:
        # Log other errors with details
        print(f"Unexpected error in single_task for {llm.name} and task id {task['id']}: {type(e).__name__}: {str(e)}")
        llm_elapsed = time.time() - llm_start
        print(f"Time elapsed since function start: {llm_elapsed:.1f}s")
        # Return result to continue processing other tasks, but mark this one as incorrect
        result.update({
            "predicted_answer": "ERROR",
            "time_elapsed": round(llm_elapsed, 1),
            "is_correct": False
        })
    
    return result

%choose multi_task
@kbench.task(name=TASK_NAME)
def multi_task(llm, df) -> float:
    print(f"Starting evaluation with {len(df)} tasks")
    print(f"LLM: {llm.name}")

    # Convert dataframe rows to list of dicts for evaluation
    tasks_list = df.to_dict('records')
    
    with kbench.client.enable_cache():
        runs = single_task.evaluate(
            stop_condition=lambda runs: len(runs) == len(tasks_list),
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            task=tasks_list,  # Pass tasks as list of dicts
            n_jobs=1,  # Use single job 
            timeout=None,  # No timeout - let it complete
            remove_run_files=True,  # Optionally remove sub runs files.
        )
    
    print(f"Evaluation complete, processing results...")
    eval_df = runs.as_dataframe()
    
    if eval_df.empty:
        print("WARNING: eval_df is empty!")
        return 0.0
    
    # Extract just the results (without framework metadata)
    results_list = eval_df.result.tolist()
    
    # Print results in JSON format
    print("\nEvaluation Results (JSON format):")
    print(json.dumps(results_list, indent=2))
    print()
    
    is_correct_series = eval_df.result.str.get("is_correct")
    print(f"Number of correct: {is_correct_series.sum()}/{len(is_correct_series)}")
    
    accuracy = float(is_correct_series.mean())  # Use float() to convert from np.float.
    print(f"Final result: {accuracy:.2%}")  
    
    return accuracy

## Load tasks from dataset

In [ ]:
with open(f"/kaggle/input/datasets/joosthazelzet/spatial-reasoning-tasks/orthographic_dataset_{DATASET_VARIANT}_tasks.json") as f:    dataset = json.load(f)
    
df_tasks = pd.DataFrame(dataset['tasks'][:TASKS_SIZE])
print(f"{len(df_tasks)} tasks loaded from dataset")

## Test single sample

In [ ]:
sample = df_tasks.iloc[0]
sample

### Preview views and prompt

In [ ]:
md_views = format_input_views(sample['input']['front'], sample['input']['top'])
#print(md_views)

In [ ]:
md_mc = format_multiple_choices(sample['multiple_choices'])
#print(md_mc)

In [ ]:
md_prompt = build_prompt(
     front_view=sample['input']['front'],
     top_view=sample['input']['top'],
     multiple_choices=sample['multiple_choices']
 )
#print(md_prompt)

In [ ]:
#single_task.run(USE_MODEL, sample)

## Evaluate the dataset

In [ ]:
multi_task.run(USE_MODEL, df_tasks)